In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("RadiomicsMaskMetricsBadDropped.csv")

In [3]:
import pandas as pd
from pandas.api.types import is_numeric_dtype
import numpy as np


In [10]:
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype

# Define a consistent seed for reproducibility
RANDOM_SEED = 42

def get_distribution_based_indices(df, col, num_points=10, samples_per_point=1, seed=RANDOM_SEED):
    # Using qcut for continuous distributions
    bins = pd.qcut(df[col], q=num_points, labels=False, duplicates='drop')
    return df.groupby(bins).sample(n=samples_per_point, random_state=seed).index.tolist()

import pandas as pd

def get_discrete_bucket_indices(df, col, num_buckets=5, samples_per_bucket=3, seed=42):
    # 1. Create equal-sized buckets based on the column's distribution
    # duplicates='drop' handles cases where many rows have the same value
    buckets = pd.qcut(df[col], q=num_buckets, labels=False, duplicates='drop')
    
    # 2. Group by these new bucket IDs and sample
    # We use a lambda to ensure we don't crash if a bucket is smaller than samples_per_bucket
    indices = (
        df.groupby(buckets, group_keys=False)
        .apply(lambda x: x.sample(n=min(len(x), samples_per_bucket), random_state=seed))
        .index
        .tolist()
    )
    
    return indices

column_tasks = {}

for col in df.columns:
    if is_numeric_dtype(df[col].dtype):
        if col != "num_components":
            # Continuous sampling based on quantiles
            indices = get_distribution_based_indices(df, col, num_points=10, seed=RANDOM_SEED)
            column_tasks[col] = df.loc[indices].copy()
            column_tasks[col]['column_task'] = col
            
        elif col == "num_components":
            # Discrete sampling based on unique values
            indices = get_discrete_bucket_indices(df, col, num_buckets=5, seed=RANDOM_SEED)
            column_tasks[col] = df.loc[indices].copy()
            column_tasks[col]['column_task'] = col

# Concatenate all into one final DataFrame
final_df = pd.concat(column_tasks.values(), ignore_index=True)

In [15]:
BOUNDED_COLUMNS = ["sphericity", "elongation", "flatness", "surface_volume_ratio"]
# --- ADD RANDOM NOISE (+/- 10%) ---
np.random.seed(RANDOM_SEED)

for col_name in final_df['column_task'].unique():
    # 1. Skip num_components entirely
    if col_name == "num_components":
        continue
        
    mask = final_df['column_task'] == col_name
    
    # Generate a uniform noise multiplier between 0.9 (-10%) and 1.1 (+10%)
    noise_multiplier = np.random.uniform(low=0.9, high=1.1, size=mask.sum())
    
    # Apply the noise
    final_df.loc[mask, col_name] = final_df.loc[mask, col_name] * noise_multiplier
    
    # 2. Clamp specific columns so they remain bounded between 0 and 1
    if col_name in BOUNDED_COLUMNS:
        final_df.loc[mask, col_name] = final_df.loc[mask, col_name].clip(lower=0.0, upper=1.0)
    
    # Round back to integer if the original column was integer-based
    elif pd.api.types.is_integer_dtype(df[col_name]):
        final_df[col_name] = final_df[col_name].round()

In [16]:
final_df

,bdmap_id,organ,diameter_x_mm,diameter_y_mm,diameter_z_mm,volume_ml,sphericity,surface_volume_ratio,elongation,flatness,max_3d_diameter_mm,num_components,column_task
0,BDMAP_00068977,gallbladder,11.802608,13.148442,7.500000,0.645249,0.770104,0.726706,0.799572,0.629965,15.009452,1,diameter_x_mm
1,BDMAP_00037592,prostate,17.547638,18.984375,7.012121,1.403719,0.778428,0.554846,0.663788,0.000000,20.371906,1,diameter_x_mm
2,BDMAP_00002119,colon,25.194566,27.890639,10.000000,4.242122,0.711873,0.419650,0.846953,0.350809,31.304721,1,diameter_x_mm
3,BDMAP_00158179,gallbladder,25.370769,24.398425,20.000000,6.285171,0.789659,0.331848,0.862563,0.718471,29.337750,1,diameter_x_mm
4,BDMAP_00166067,gallbladder,28.839192,52.154297,29.000000,8.592632,0.283780,0.832008,0.455253,0.358024,57.959780,1,diameter_x_mm
...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,BDMAP_00171970,uterus,19.042969,17.578125,19.000000,2.510436,0.554865,0.641279,0.753903,0.685382,24.769887,1,num_components
92,BDMAP_00181011,uterus,7.988281,7.988281,5.000000,0.214038,0.823027,0.982290,0.985236,0.735583,9.421560,1,num_components
93,BDMAP_00014624,esophagus,110.601562,140.765625,225.000000,73.160137,0.372668,0.310268,0.185093,0.146582,228.604679,3,num_components
94,BDMAP_00182906,duodenum,19.855469,18.992188,24.000000,1.250413,0.584828,0.767547,0.322351,0.280190,29.233632,2,num_components


In [13]:
final_df.to_csv("EvaluationSpacedData.csv",index=False)

In [1]:
import pandas as pd

In [10]:
temp = pd.read_csv("AAProTrain_V7.csv")

In [11]:
ld = pd.read_csv("AAProTrain.csv")

In [14]:
temp.shape

(1786, 15)

In [13]:
temp = temp[temp["bdmap_id"].isin(ld["bdmap_id"])]

In [15]:
temp.to_csv("AAProTrain_V7.csv",index=False)